This notebook goes through the entire process that, starting from breast MRI images and the corresponding binary masks identifying the tumor region, leads to the automatic classification of a specific diagnostic or prognostic clinical variable.

Lista degli import che vengono fatti mano a mano:
- import os
- import nibabel as nib

In [14]:
import os

# Ask the user to enter the path to the main dataset folder
dataset_path = input("Enter the path to the main dataset folder, for example, './images': ")

# Check if the path exists
if not os.path.isdir(dataset_path):
    print("Error: the entered path is not valid.")
else:
    print(f"Folder found: {dataset_path}")
    print("The expected folder structure is as follows:")
    print("main_dataset_folder/")
    print("├── Patient_XXX/")
    print("│   ├── Patient_XXX_000Y.nii")
    print("└...")
    print("Where XXX is the patient ID and 000Y is the channel identifier of the image, following the nnUNet dataset format.")

    # Get the list of patient subfolders inside the main dataset folder
    patient_list = [name for name in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, name))]
    print(f"{len(patient_list)} patient folders found.")
    print("The list of patient IDs has been stored in the 'patient_list' variable.")
    # print(patient_list)

Enter the path to the main dataset folder:  .\images


Folder found: .\images
The expected folder structure is as follows:
main_dataset_folder/
├── Patient_XXX/
│   ├── Patient_XXX_000Y.nii
└...
Where XXX is the patient ID and 000Y is the channel identifier of the image, following the nnUNet dataset format.
3 patient folders found.
The list of patient IDs has been stored in the 'patient_list' variable.


In [13]:
import nibabel as nib

# Ask the user to input a specific patient ID
patient_id = input("Enter the patient ID you want to inspect (as in the folder name): ")

# Create the full path to the patient folder
patient_folder = os.path.join(dataset_path, patient_id)

# Check if the folder exists
if not os.path.isdir(patient_folder):
    print("Error: patient folder not found.")
else:
    print(f"Patient folder found: {patient_folder}")
    
    # List all NIfTI files in the patient folder
    nifti_files = [f for f in os.listdir(patient_folder) if f.endswith(".nii") or f.endswith(".nii.gz")]
    
    if len(nifti_files) == 0:
        print("No NIfTI files found in this folder.")
    else:
        print(f"{len(nifti_files)} NIfTI file(s) found:")
        
        for filename in nifti_files:
            file_path = os.path.join(patient_folder, filename)
            try:
                nifti_img = nib.load(file_path)
                data = nifti_img.get_fdata()
                header = nifti_img.header
                pixdim = header.get_zooms()
                
                # Typically: pixdim[0]=qfac (unused), [1]=x spacing, [2]=y spacing, [3]=slice thickness
                pixel_spacing = pixdim[:2]
                slice_thickness = pixdim[2] if len(pixdim) > 2 else "Unknown"
                
                print(f"  - {filename}, shape: {data.shape}, dtype: {data.dtype}, pixel spacing: {pixel_spacing} mm, slice thickness : {slice_thickness} mm")
            except Exception as e:
                print(f"  - {filename}: could not be loaded ({e})\n")

Enter the patient ID you want to inspect (as in the folder name):  DUKE_001


Patient folder found: .\images\DUKE_001
5 NIfTI file(s) found:
  - DUKE_001_0000.nii.gz, shape: (448, 448, 160), dtype: float64, pixel spacing: (0.8035714, 0.8035714) mm, slice thickness : 1.100000023841858 mm
  - DUKE_001_0001.nii.gz, shape: (448, 448, 160), dtype: float64, pixel spacing: (0.8035714, 0.8035714) mm, slice thickness : 1.100000023841858 mm
  - DUKE_001_0002.nii.gz, shape: (448, 448, 160), dtype: float64, pixel spacing: (0.8035714, 0.8035714) mm, slice thickness : 1.100000023841858 mm
  - DUKE_001_0003.nii.gz, shape: (448, 448, 160), dtype: float64, pixel spacing: (0.8035714, 0.8035714) mm, slice thickness : 1.100000023841858 mm
  - DUKE_001_0004.nii.gz, shape: (448, 448, 160), dtype: float64, pixel spacing: (0.8035714, 0.8035714) mm, slice thickness : 1.100000023841858 mm


## pyradiomics